# Practical 07: Search Strategies and Propositional Logic

## Part A – Uninformed Search Algorithms 

### Task 01 – Breadth-First Search (BFS) with Shortest Path Return 
#### Program 1: BFS Returning Shortest Path

Breadth-First Search (BFS) is a graph traversal algorithm that explores nodes level by level starting from a given source node. 

Instead of going deep into one branch, it first visits all the immediate neighbors of the starting node, then moves to the next level of neighbors. 

BFS uses a queue (FIFO – First In First Out) to manage the order of exploration and a visited set to avoid revisiting nodes. 

In unweighted graphs, BFS is commonly used to find the shortest path because the first time the goal node is reached, it guarantees the minimum number of edges from the start node.

In [4]:
from collections import deque 

def bfs_shortest_path(graph, start, goal): 

    # A set to store visited nodes (to avoid revisiting)
    visited = set() 

    # Queue stores tuples: (current_node, path_taken)
    queue = deque([(start, [start])])   

    while queue: 
        # Remove first element from queue (FIFO)
        node, path = queue.popleft()    
        
        if node == goal: 
            # Return the shortest path immediately
            return path      
        
        if node not in visited: 
            visited.add(node) 
                
            # Check all adjacent nodes    
            for neighbor in graph[node]:   
                
                # Add neighbor to queue
                # Update path by adding neighbor to current path
                queue.append((neighbor, path + [neighbor]))     
                

    return None 

graph = { 
    'A': ['B', 'C'], 
    'B': ['D', 'E'], 
    'C': ['F'], 
    'D': [], 
    'E': ['G'], 
    'F': [], 
    'G': [] 
} 

print("Shortest Path:", bfs_shortest_path(graph, 'A', 'G')) 

Shortest Path: ['A', 'B', 'E', 'G']


### Task 02 – BFS in 2D Grid Maze

Breadth-First Search (BFS) in a 2D grid maze is used to find the shortest path from a start position to a goal position. 

The grid consists of 0s (open paths) and 1s (blocked cells). The algorithm explores the maze level by level using a queue and checks four possible directions (up, down, left, right).

In [5]:
from collections import deque 
    
def bfs_grid(grid, start, goal): 

    # Get number of rows and columns in the grid
    rows, cols = len(grid), len(grid[0]) 

    # Queue stores ((x, y), path_taken)
    queue = deque([(start, [start])]) 
    visited = set([start]) 
 
    # Possible movements: Right, Down, Left, Up
    directions = [(0,1),(1,0),(0,-1),(-1,0)] 
 
    while queue: 
        # Remove the first element from queue (FIFO)
        (x, y), path = queue.popleft() 
 
        if (x, y) == goal: 
            return path 
 
        for dx, dy in directions: 
            # Calculate next position
            nx, ny = x+dx, y+dy 
 
            # Check if next position is inside grid boundaries
            if 0 <= nx < rows and 0 <= ny < cols: 

                # Check if cell is open (0) and not visited
                if grid[nx][ny] == 0 and (nx, ny) not in visited: 
                    visited.add((nx, ny)) 

                    # Add new cell to queue with updated path
                    queue.append(((nx, ny), path + [(nx, ny)]))

    return None

grid = [ 
 [0, 1, 0, 0], 
 [0, 0, 0, 1], 
 [1, 0, 1, 0], 
 [0, 0, 0, 0] 
] 
 
path = bfs_grid(grid, (0,0), (3,3)) 
print("Grid Path:", path) 
print("Path Length:", len(path) if path else 0)

Grid Path: [(0, 0), (1, 0), (1, 1), (2, 1), (3, 1), (3, 2), (3, 3)]
Path Length: 7


### Task 03 – Depth-First Search (Iterative) 

Depth-First Search (DFS) explores a graph by going as deep as possible along each branch before backtracking. 

Iterative DFS uses a stack instead of recursion. 

The algorithm pops a node from the stack, marks it visited, and pushes all its unvisited neighbors. By using a stack, DFS goes deep along one path, backtracks when necessary, and eventually visits all reachable nodes. (LIFO)

In [ ]:
def dfs_iterative(graph, start): 
    visited = set() 
    stack = [start] 
 
    while stack: 
        # Remove the last element from stack (LIFO)
        node = stack.pop() 
 
        if node not in visited: 
            print(node, end=" ") 
            visited.add(node) 
            
            # reversed() is used to maintain the order of neighbors
            # so that left-most neighbor is visited first
            stack.extend(reversed(graph[node]))

graph = { 
    'A': ['B', 'C'], 
    'B': ['D', 'E'], 
    'C': ['F'], 
    'D': [], 
    'E': ['G'], 
    'F': [], 
    'G': [] 
} 

print("DFS Traversal:") 
dfs_iterative(graph, 'A')

DFS Traversal:
A B D E G C F 

### Task 04 – Cycle Detection using DFS

This function detects cycles in a graph using Depth-First Search (DFS). 

It maintains two sets: visited to track nodes already fully explored, and rec_stack to track nodes in the current DFS path. 

If during DFS a neighbor is found that is already in the recursion stack, it means a back edge exists, which forms a cycle. The function explores all nodes to handle disconnected components, and returns True if any cycle is detected.

In [ ]:
def detect_cycle(graph):
    visited = set()      
    rec_stack = set()    

    # Helper function: DFS traversal
    def dfs(node):
        # Mark current node as visited
        visited.add(node)  
        # Add node to recursion stack    
        rec_stack.add(node)    

        for neighbor in graph[node]:
            if neighbor not in visited:
                # Recurse on unvisited neighbors
                if dfs(neighbor):     
                    return True
                
            # Back edge found → cycle
            elif neighbor in rec_stack:   
                return True
            
        # Remove node from recursion stack before returning
        rec_stack.remove(node)   
        return False

    # Check all nodes (handles disconnected graphs)
    for node in graph:
        if node not in visited:
            if dfs(node):
                return True
            
    # No cycle found
    return False  

graph = {
    'A': ['B'],
    'B': ['C'],
    'C': ['D'],
    'D': ['B']  # Back edge creating a cycle B -> C -> D -> B
}
print("Cycle Exists:", detect_cycle(graph))

Cycle Exists: True


## Part B – Informed Search Algorithms
### Task 05 – Greedy Best-First Search

Uses a heuristic function to estimate the cost from a node to the goal.

It selects the node with the lowest heuristic value at each step. 

GBFS uses a priority queue to choose which node to explore next and keeps track of visited nodes to avoid loops. 

When the goal is reached, the path is reconstructed using a parent dictionary. 

In [ ]:
import heapq
from platform import node
def greedy_best_first(graph, start, goal, heuristic): 
    visited = set() 

    # Priority queue: (heuristic value, node)
    pq = [(heuristic[start], start)] 
    # To reconstruct path
    parent = {start: None} 

    while pq: 
        # Get node with smallest heuristic
        _, node = heapq.heappop(pq) 
        
        if node in visited: 
            continue 
        
        visited.add(node) 
        
        if node == goal: 
            path = [] 

            # Reconstruct path using parent dictionary
            while node: 
                path.append(node) 
                node = parent[node] 

             # Reverse path to get start -> goal
            return path[::-1] 
        
        for neighbor in graph[node]: 
            if neighbor not in visited: 

                # Add neighbor to priority queue
                heapq.heappush(pq, (heuristic[neighbor], neighbor)) 
            if neighbor not in parent: 
                # Track parent for path reconstruction
                parent[neighbor] = node 
    return None 

graph = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F'],
    'D': ['G'],
    'E': ['G'],
    'F': ['G'],
    'G': []
}

heuristic = {
    'A': 5,
    'B': 2,
    'C': 4,
    'D': 1,
    'E': 6,
    'F': 2,
    'G': 0
}

start_node = 'A'
goal_node = 'G'

path = greedy_best_first(graph, start_node, goal_node, heuristic)
print("Greedy Best-First Search Path:", path)

Greedy Best-First Search Path: ['A', 'B', 'D', 'G']


### Task 06 – A* Search Algorithm

A* Search Algorithm is an informed search that finds the shortest path from a start node to a goal node in a weighted graph. It uses two values:

g(n) -> actual cost from start to the current node

h(n) -> estimated cost from the current node to the goal (heuristic)

A* chooses the node with the lowest f(n) = g(n) + h(n) from the priority queue. It keeps track of parents to reconstruct the path. 

In [13]:
import heapq

def a_star(graph, start, goal, heuristic): 
    open_list = [] 
    # Priority queue for nodes to explore: stores (f_cost, node) 
    # Start node with f_cost = 0
    heapq.heappush(open_list, (0, start)) 

    # Actual cost from start to node
    g_cost = {start: 0} 
    # To reconstruct path
    parent = {start: None} 

    while open_list: 
        # Node with lowest f_cost = g_cost + heuristic
        _, current = heapq.heappop(open_list) 
    
        if current == goal: 
            path = [] 
            while current: 
                path.append(current) 
                current = parent[current] 

            # Return path from start → goal
            return path[::-1] 
    
        for neighbor, cost in graph[current]: 
            # g(n) = cost to reach neighbor
            new_cost = g_cost[current] + cost 
        
            # If neighbor is unvisited or found a cheaper path
            if neighbor not in g_cost or new_cost < g_cost[neighbor]: 
                g_cost[neighbor] = new_cost 

                # f(n) = g(n) + h(n)
                f_cost = new_cost + heuristic[neighbor] 
                heapq.heappush(open_list, (f_cost, neighbor)) 
                parent[neighbor] = current 
    return None 

weighted_graph = { 
'A': [('B',1), ('C',3)], 
'B': [('D',1), ('E',5)], 
'C': [('F',2)], 
'D': [], 
'E': [('G',1)], 
'F': [], 
'G': [] 
} 

heuristic = { 
    'A':5, 'B':2, 'C':4, 'D':1, 'E':6, 'F':2, 'G':0 
}

print("A* Path:", a_star(weighted_graph, 'A', 'G', heuristic))
   

A* Path: ['A', 'B', 'E', 'G']


## Part C – Hill Climbing 

Hill Climbing is a local search algorithm used to find a solution by continuously moving toward the neighbor with the highest value (or lowest cost). '

It starts from an initial state and chooses the next state based on a heuristic function that estimates “goodness.” 

The process repeats until it reaches a peak (local maximum) where no neighbor is better.

### Task 07 – Simple Hill Climbing

Algorithm moves to the first neighboring state that has a better evaluation 
than the current state. 

Once a better neighbor is found, it immediately moves there, and the process 
repeats.

In [15]:
def hill_climbing(function, start, step=0.1, max_iter=100): 
    # Starting point
    current = start 

    for _ in range(max_iter): 
        # Generate neighbors by moving a small step forward and backward
        neighbors = [current + step, current - step] 

        # Choose the neighbor with the highest function value
        next_state = max(neighbors, key=function) 

        # If no improvement, return current (local maximum)
        if function(next_state) <= function(current): 
            return current 
        
        current = next_state 
    return current 

def f(x):
    return -(x-2)**2 + 4  # Maximum at x=2, f(2)=4

# Starting point
start_point = 0  
max_value = hill_climbing(f, start_point, step=0.5, max_iter=100)

# stops when no neighbor is better, which is your local maximum
print("Local Maximum at x =", max_value)
print("Maximum Value f(x) =", f(max_value))

Local Maximum at x = 2.0
Maximum Value f(x) = 4.0


### Task 08 – Steepest Ascent Hill Climbing

Evaluates all neighboring states and then selects the one with the best evaluation (the steepest ascent). 

Instead of checking only two neighbors (current ± step), it checks many neighbors in a small range around the current state.

This approach ensures that the algorithm always makes the best possible move in terms of the objective function at each step.


In [16]:
def steepest_ascent(function, start, step=0.1, max_iter=100): 
    current = start 

    for _ in range(max_iter): 
        # Generate neighbors: small steps around current position
        # Here, 11 neighbors: current ± 5*step
        neighbors = [current + i*step for i in range(-5,6)] 

        # Choose neighbor with highest function value
        next_state = max(neighbors, key=function) 
        
        # Stop if no improvement
        if function(next_state) <= function(current): 
            break 
        
        current = next_state 
    return current 

def f(x):
    return -(x-2)**2 + 4  # Maximum at x=2

result = steepest_ascent(f, start=0) 
print("Steepest Ascent Result:", result)

Steepest Ascent Result: 2.0


## Part D – Simulated Annealing 

Probabilistic optimization algorithm

Used to find a global optimum (maximum or minimum) of a function, even when the function has many local maxima or minima.


### Task 09 – Simulated Annealing (Function Optimization)

In [ ]:
import math 
import random


def simulated_annealing(function, start, temp=100, cooling=0.95, max_iter=1000): 
    # Current solution
    current = start 
    # Best solution found so far
    best = current 

    for i in range(max_iter): 
         # Pick a neighbor randomly within [-1,1] range
        neighbor = current + random.uniform(-1,1) 

        # Calculate difference in function value
        delta = function(neighbor) - function(current) 

        # Decide whether to move to neighbor
        # If delta > 0 -> neighbor is better, move there
        # If delta < 0 -> neighbor is worse, move there with probability exp(delta/temp)

        if delta > 0 or random.random() < math.exp(delta/temp): 
            current = neighbor 
    
        if function(current) > function(best): 
            best = current 
        
        temp *= cooling 
    return best 

def f(x):
    return -(x-2)**2 + 4  # Maximum at x=2

result = simulated_annealing(f, start=random.uniform(-5,5)) 
print("Simulated Annealing Result:", result) 
print("Function Value:", f(result)) 

Simulated Annealing Result: 1.999562260583871
Function Value: 3.9999998083842034


### Task 11 – Implement Uniform Cost Search (UCS)

Uniform Cost Search (UCS) is a graph search algorithm that finds the least-cost path from a start node to a goal node. 

It uses a priority queue to always expand the node with the lowest cumulative cost. 

Expands cheapest path first, unlike BFS which uses level-order traversal.


In [18]:
def uniform_cost_search(graph, start, goal): 
    # Priority queue: (cost to reach node, node)
    pq = [(0, start)] 
    visited = set() 
    # Keep track of path
    parent = {start: None} 
    # Cost to reach each node
    cost = {start: 0} 

    while pq: 
        # Pop node with smallest cost
        curr_cost, node = heapq.heappop(pq) 
        if node == goal: 
            path = [] 

            while node: 
                # Reconstruct path from goal -> start
                path.append(node) 
                node = parent[node] 

            # Reverse path to get start → goal
            return path[::-1] 
        
        if node not in visited: 
            visited.add(node) 
            
            for neighbor, weight in graph[node]: 
                # Total cost to reach neighbor
                new_cost = curr_cost + weight 
                
                if neighbor not in cost or new_cost < cost[neighbor]: 
                    cost[neighbor] = new_cost 
                    heapq.heappush(pq, (new_cost, neighbor)) 
                    parent[neighbor] = node 
    return None 

weighted_graph = {
    'A': [('B',1), ('C',3)],
    'B': [('D',1), ('E',5)],
    'C': [('F',2)],
    'D': [],
    'E': [('G',1)],
    'F': [],
    'G': []
}

print("UCS Path:", uniform_cost_search(weighted_graph, 'A', 'G')) 

UCS Path: ['A', 'B', 'E', 'G']


## Part E Extension – Depth First Search (DFS) 

### Task 12 – DFS (Recursive) Returning Path 

In [2]:
def dfs_path(graph, start, goal, path=None, visited=None): 
    if visited is None: 
        visited = set()
    if path is None: 
        path = [] 
        
    visited.add(start) 
    # Add current node to path
    path = path + [start] 
        
    if start == goal: 
        return path 
 
    # Explore neighbors recursively
    for neighbor in graph[start]: 
        if neighbor not in visited: 
            result = dfs_path(graph, neighbor, goal, path, visited) 
                
            # If path found, return it immediately
            if result: 
                return result 
                
    # No path found from this branch
    return None 
 
 
graph = { 
    'A': ['B', 'C'], 
    'B': ['D', 'E'], 
    'C': ['F'], 
    'D': [], 
    'E': ['G'], 
    'F': [], 
    'G': [] 
} 
 
print("DFS Path:", dfs_path(graph, 'A', 'G'))

DFS Path: ['A', 'B', 'E', 'G']


### Task 13 – DFS Iterative Returning Path

In [3]:
def dfs_iterative_path(graph, start, goal): 
    # Stack stores tuples: (current_node, path_so_far)
    stack = [(start, [start])] 
    # Keep track of visited nodes to avoid cycles
    visited = set() 
 
    while stack: 
        # Take the top node from the stack
        node, path = stack.pop() 
 
        if node == goal: 
            return path 
 
        if node not in visited: 
            visited.add(node) 

            # Add neighbors to stack
            # reversed() ensures the first neighbor in graph is visited first
            for neighbor in reversed(graph[node]): 
                stack.append((neighbor, path + [neighbor])) 
 
    return None 

graph = { 
    'A': ['B', 'C'], 
    'B': ['D', 'E'], 
    'C': ['F'], 
    'D': [], 
    'E': ['G'], 
    'F': [], 
    'G': [] 
} 

print("DFS Iterative Path:", dfs_iterative_path(graph, 'A', 'G'))


DFS Iterative Path: ['A', 'B', 'E', 'G']


### Task 13 – DFS on 2D Grid Maze

In [ ]:
def dfs_grid(grid, start, goal): 
    rows, cols = len(grid), len(grid[0]) 

    # Stack stores tuples: ((x, y), path_so_far)
    stack = [(start, [start])] 
    visited = set([start]) 

    # Define 4 possible movement directions: right, down, left, up
    directions = [(0,1),(1,0),(0,-1),(-1,0)] 
    
    while stack: 
        (x, y), path = stack.pop() 
        
        if (x, y) == goal: 
            return path 

        # Explore all neighbors
        for dx, dy in directions: 
            nx, ny = x+dx, y+dy 

            # Check boundaries: must be inside the grid
            if 0 <= nx < rows and 0 <= ny < cols: 
                if grid[nx][ny] == 0 and (nx, ny) not in visited: 
                    visited.add((nx, ny)) 
                    stack.append(((nx, ny), path + [(nx, ny)])) 
    return None 

grid = [ 
[0, 1, 0, 0], 
[0, 0, 0, 1], 
[1, 0, 1, 0], 
[0, 0, 0, 0] 
] 

print("DFS Grid Path:", dfs_grid(grid, (0,0), (3,3))) 

DFS Grid Path: [(0, 0), (1, 0), (1, 1), (2, 1), (3, 1), (3, 2), (3, 3)]


### Task 14 – DFS with Depth Limit (Depth-Limited Search)

In [4]:
def depth_limited_dfs(graph, start, goal, limit, path=None): 
    if path is None: 
        # Initialize path with start node
        path = [start] 

    if start == goal: 
        return path 
    
    # Limit check: stop recursion if depth limit reached
    if limit <= 0:
        return None 
 
    # Explore neighbors recursively
    for neighbor in graph[start]: 
        if neighbor not in path: 
            result = depth_limited_dfs( 
                graph, 
                neighbor, 
                goal, 
                limit-1, # Reduce remaining depth
                path + [neighbor] # Reduce remaining depth
            ) 
            if result: 
                return result 
 
    # No path found within depth limit
    return None 
 
graph = { 
    'A': ['B', 'C'], 
    'B': ['D', 'E'], 
    'C': ['F'], 
    'D': [], 
    'E': ['G'], 
    'F': [], 
    'G': [] 
} 
print("Depth Limited DFS:", depth_limited_dfs(graph, 'A', 'G', 2))

Depth Limited DFS: None


### Task 15 – Iterative Deepening DFS (IDDFS)

Iterative Deepening DFS (IDDFS) combines the benefits of DFS and BFS:

Performs Depth-Limited DFS repeatedly with increasing depth limits.

Finds the shortest path like BFS but uses less memory like DFS.

Useful when the maximum depth of the solution is unknown.

In [25]:
def depth_limited_dfs(graph, start, goal, limit, path=None): 
    if path is None: 
        # Initialize path with start node
        path = [start] 

    if start == goal: 
        return path 
    
    # Limit check: stop recursion if depth limit reached
    if limit <= 0:
        return None 
 
    # Explore neighbors recursively
    for neighbor in graph[start]: 
        if neighbor not in path: 
            result = depth_limited_dfs( 
                graph, 
                neighbor, 
                goal, 
                limit-1, # Reduce remaining depth
                path + [neighbor] # Reduce remaining depth
            ) 
            if result: 
                return result 
 
    # No path found within depth limit
    return None 
 
def iterative_deepening_dfs(graph, start, goal, max_depth): 
    
    # Try DFS with increasing depth limits from 0 up to max_depth
    for depth in range(max_depth + 1): 
        result = depth_limited_dfs(graph, start, goal, depth) 
        if result: 
            return result 
    return None 
 
graph = { 
    'A': ['B', 'C'], 
    'B': ['D', 'E'], 
    'C': ['F'], 
    'D': [], 
    'E': ['G'], 
    'F': [], 
    'G': [] 
} 

print("IDDFS Path:", iterative_deepening_dfs(graph, 'A', 'G', 5))

IDDFS Path: ['A', 'B', 'E', 'G']


### Task 16 – DFS Cycle Detection (Directed Graph)

In [2]:
def detect_cycle_dfs(graph): 
    visited = set() 
    # Track nodes in the current DFS path
    recursion_stack = set() 
 
    def dfs(node): 
        visited.add(node) 
        # Add node to recursion stack
        recursion_stack.add(node) 
 
        for neighbor in graph[node]: 
            if neighbor not in visited: 
                if dfs(neighbor): 
                    return True 
            
            # Cycle found
            elif neighbor in recursion_stack: 
                return True 
 
        recursion_stack.remove(node) 
        return False 
 
    for node in graph: 
        if node not in visited: 
            if dfs(node): 
                return True 
 
    return False 

# graph = { 
#     'A': ['B', 'C'], 
#     'B': ['D', 'E'], 
#     'C': ['F'], 
#     'D': [], 
#     'E': ['G'], 
#     'F': [], 
#     'G': [] 
# }  

graph = {
    'A': ['B'],
    'B': ['C'],
    'C': ['D'],
    'D': ['B']  # Back edge creating a cycle B -> C -> D -> B
}
 
print("Cycle Exists:", detect_cycle_dfs(graph))


Cycle Exists: True


### Task 17 – Topological Sort Using DFS (Directed Acyclic Graph (DAG) Only)

Topological sort only works on DAGs because cycles would make ordering impossible.

Directed    - All edges have a direction (like A -> B).

Acyclic     - There are no cycles (you cannot start at a node and follow edges to come back to the same node).

A → C

B → C

C → D

A must be done before C

B must be done before C

C must be done before D

In [ ]:
def topological_sort(graph): 
    visited = set() 
    stack = [] 
 
    def dfs(node): 
        visited.add(node) 
        for neighbor in graph[node]: 
            if neighbor not in visited: 
                dfs(neighbor) 
        stack.append(node) 
 
    for node in graph: 
        if node not in visited: 
            dfs(node) 
 
    return stack[::-1] 
 
 
dag = { 
    'A': ['C'], 
    'B': ['C', 'D'], 
    'C': ['E'], 
    'D': ['F'], 
    'E': [], 
    'F': [] 
} 
 
print("Topological Order:", topological_sort(dag))

Topological Order: ['B', 'D', 'F', 'A', 'C', 'E']


### Task 18 – DFS Counting Nodes Expanded

DFS with Metrics not only finds a path from start to goal but also counts the number of nodes expanded during the search.

Stack ensures DFS explores deepest nodes first.

nodes_expanded helps measure search efficiency.

Useful in AI/search problems to compare algorithm performance.

In [29]:
def dfs_with_metrics(graph, start, goal): 

    # Stack stores tuples: (current_node, path_so_far)
    stack = [(start, [start])] 
    visited = set() 
    # Counter for nodes expanded
    nodes_expanded = 0

    while stack: 
        # Pop the current node and path
        node, path = stack.pop() 
        # Increment nodes expanded
        nodes_expanded += 1 

        if node == goal: 
            return path, nodes_expanded 

        # Explore neighbors
        if node not in visited: 
            visited.add(node) 
            for neighbor in reversed(graph[node]): 
                stack.append((neighbor, path + [neighbor])) 

    return None, nodes_expanded 

graph = { 
    'A': ['B', 'C'], 
    'B': ['D', 'E'], 
    'C': ['F'], 
    'D': [], 
    'E': ['G'], 
    'F': [], 
    'G': [] 
}  

path, count = dfs_with_metrics(graph, 'A', 'G') 
print("Path:", path) 
print("Nodes Expanded:", count)

Path: ['A', 'B', 'E', 'G']
Nodes Expanded: 5


### Task 19 – DFS Printing All Root-to-Leaf Paths (Game Tree) 

DFS All Paths explores every path from the start node to all leaves in a tree or DAG.

In [2]:
def dfs_all_paths(graph, start, path=None): 
    if path is None: 
        path = [start] 

    if not graph[start]: 
        print("Path:", path) 
        return 
    
    for neighbor in graph[start]: 
        dfs_all_paths(graph, neighbor, path + [neighbor]) 

game_tree = { 
'S': ['A', 'B'], 
'A': ['C', 'D'], 
'B': ['E'], 
'C': [], 
'D': [], 
'E': [] 
} 

dfs_all_paths(game_tree, 'S')

Path: ['S', 'A', 'C']
Path: ['S', 'A', 'D']
Path: ['S', 'B', 'E']
